**Задание 1. Реализация PWM с нуля.**

In [2]:
sites = ["GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA",
         "ACAGTCAGC", "TAGGTCAGC", "CAGGTCAGC",
         "CAGGTCGAT", "CAGGTCAGC", "CAGGTCAGC",
         "CAGGTTGGC"]


Ф-ция, возвращающая PFM и сама рассчитанная PFM:

In [5]:
import numpy as np

def seqs_to_pfm(sequences, nucs='ATGC'):
    '''
    Ф-ция, возвращающая PFM
    '''
    n = len(sequences[0])
    pfm = np.zeros((len(nucs), n))
    for seq in sequences:
        for i, base in enumerate(seq):
            pfm[nucs.index(base), i] += 1
    return pfm

pfm = seqs_to_pfm(sites)

In [10]:
import pandas as pd

df_pfm = pd.DataFrame(pfm, index=[i for i in 'ATGC'], columns=[i for i in range(len(sites[0]))])

print('PFM:')
print(df_pfm)

PFM:
     0    1    2     3     4    5    6    7    8
A  1.0  8.0  1.0   0.0   0.0  2.0  7.0  2.0  1.0
T  2.0  0.0  0.0   0.0  10.0  2.0  0.0  0.0  1.0
G  1.0  0.0  8.0  10.0   0.0  0.0  3.0  8.0  0.0
C  6.0  2.0  1.0   0.0   0.0  6.0  0.0  0.0  8.0


Переход PFM -> PPM c псевдосчетом $\alpha = 0.1$:

In [7]:
def pfm_to_ppm(pfm, alpha):
    return (pfm + alpha) / (pfm.sum(axis=0) + 4 * alpha)

alpha = 0.1
ppm = pfm_to_ppm(pfm, alpha)

In [12]:
df_ppm = pd.DataFrame(ppm, index=[i for i in 'ATGC'], columns=[i for i in range(len(sites[0]))])

print('PPM:')
print(df_ppm)

PPM:
          0         1         2         3         4         5         6  \
A  0.105769  0.778846  0.105769  0.009615  0.009615  0.201923  0.682692   
T  0.201923  0.009615  0.009615  0.009615  0.971154  0.201923  0.009615   
G  0.105769  0.009615  0.778846  0.971154  0.009615  0.009615  0.298077   
C  0.586538  0.201923  0.105769  0.009615  0.009615  0.586538  0.009615   

          7         8  
A  0.201923  0.105769  
T  0.009615  0.105769  
G  0.778846  0.009615  
C  0.009615  0.778846  


Переход PPM -> PWM c фоном генома: P(A) = P(T) = 0.295, P(G) = P(C) = 0.205:

In [9]:
background = np.array([0.295, 0.295, 0.205, 0.205]) #P(A), P(T), P(G), P(C)

def ppm_to_pwm(ppm, background):
    return np.log2(ppm / background[:, np.newaxis])

pwm = ppm_to_pwm(ppm, background)

In [13]:
df_pwm = pd.DataFrame(pwm, index=[i for i in 'ATGC'], columns=[i for i in range(len(sites[0]))])

print('PWM:')
print(df_pwm)

PWM:
          0         1         2         3         4         5         6  \
A -1.479795  1.400623 -1.479795 -4.939227 -4.939227 -0.546909  1.210521   
T -0.546909 -4.939227 -4.939227 -4.939227  1.718985 -0.546909 -4.939227   
G -0.954704 -4.414136  1.925714  2.244076 -4.414136 -4.414136  0.540061   
C  1.516602 -0.021818 -0.954704 -4.414136 -4.414136  1.516602 -4.414136   

          7         8  
A -0.546909 -1.479795  
T -4.939227 -1.479795  
G  1.925714 -4.414136  
C -4.414136  1.925714  


Найдем max и min теоретически возможные скоры для нашей PWM, а также посл-ти их дающие:

In [16]:
nucs='ATGC'

max_score_per_pos = pwm.max(axis=0)
min_score_per_pos = pwm.min(axis=0)
max_score = max_score_per_pos.sum()
min_score = min_score_per_pos.sum()

max_seq = ''.join([nucs[pwm[:, i].argmax()] for i in range(pwm.shape[1])])
min_seq = ''.join([nucs[pwm[:, i].argmin()] for i in range(pwm.shape[1])])

print("Max score:", max_score, "Sequence of max score:", max_seq)
print("Min score:", min_score, "Sequence of min score:", min_seq)

Max score: 15.384551840581208 Sequence of max score: CAGGTCAGC
Min score: -39.943425491429075 Sequence of min score: ATTAAGTTG
